In [7]:
import sys
import os
import random
from typing import List

import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Add your src directory if needed
sys.path.append(os.path.abspath("../src"))

from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.instantiation import instantiate_3d
from abstractions3d.dsl.nodes import Rect3D, Move3D, SymTrans3D, SymRef3D, Union3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.data.blueprint import Table3D, Chair3D, Bench3D

random.seed(42)
torch.manual_seed(42)

In [8]:
def sample_uniform(low: float, high: float) -> float:
    return random.uniform(low, high)

def generate_dataset(num_samples: int = 50) -> List[Union3D]:
    dataset = []
    for _ in range(num_samples):
        dataset.append(Table3D(
            top_length=sample_uniform(3.0, 6.0),
            top_depth=sample_uniform(1.5, 3.0),
            top_thickness=sample_uniform(0.1, 0.3),
            leg_length=sample_uniform(0.2, 0.4),
            leg_depth=sample_uniform(0.2, 0.4),
            leg_height=sample_uniform(1.0, 2.0)
        ))
        dataset.append(Chair3D(
            seat_length=sample_uniform(1.0, 2.0),
            seat_depth=sample_uniform(1.0, 2.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=sample_uniform(1.0, 2.0),
            backrest_thickness=sample_uniform(0.1, 0.3)
        ))
        backrest_height = sample_uniform(0.0, 1.5)
        dataset.append(Bench3D(
            seat_length=sample_uniform(2.0, 5.0),
            seat_depth=sample_uniform(0.5, 1.0),
            seat_thickness=sample_uniform(0.2, 0.5),
            leg_length=sample_uniform(0.1, 0.3),
            leg_depth=sample_uniform(0.1, 0.3),
            leg_height=sample_uniform(0.5, 1.0),
            backrest_height=backrest_height,
            backrest_thickness=sample_uniform(0.0, 0.3) if backrest_height > 0 else 0.0
        ))
    return dataset

dataset = generate_dataset(50)
print(f"Generated {len(dataset)} shapes.")

Generated 150 shapes.


In [9]:
class NodeEncoder(nn.Module):
    """Encodes a single node's parameters + child latents."""
    def __init__(self, param_dim: int, child_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(param_dim + child_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, param_dim + child_dim)
        )

    def forward(self, params: torch.Tensor, child_latent: torch.Tensor):
        x = torch.cat([params, child_latent], dim=-1)
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

class RecursiveAbstraction:
    """Recursive abstraction for Shape3D trees."""
    def __init__(self, latent_dim: int = 32, lr: float = 1e-3):
        self.latent_dim = latent_dim
        self.lr = lr
        self.node_encoders = {}  # dict[node_type] -> NodeEncoder
        self.optimizers = {}

    def get_params_tensor(self, node) -> torch.Tensor:
        """Convert node parameters to tensor (float + categorical encoding)."""
        if hasattr(node, "param_tuple"):
            _, params = node.param_tuple()
            flat = []
            for p in params:
                if isinstance(p, (int, float)):
                    flat.append(float(p))
                elif isinstance(p, str):
                    flat.append({"x":0.0, "y":1.0, "z":2.0}.get(p, -1.0))
                elif isinstance(p, Shape3D):
                    flat.append(0.0)  # placeholder for child
                else:
                    flat.append(0.0)
            return torch.tensor(flat, dtype=torch.float32)
        return torch.zeros(1)

    def encode_node(self, node) -> torch.Tensor:
        """Recursively encode node + children."""
        # First, encode children
        child_latents = []
        for child in getattr(node, "children", []):
            child_latents.append(self.encode_node(child))
        child_latent = torch.cat(child_latents) if child_latents else torch.zeros(0)

        # Convert node params
        params = self.get_params_tensor(node)
        param_dim = params.numel()
        child_dim = child_latent.numel()

        node_type = type(node).__name__

        # Initialize encoder if not exists
        if node_type not in self.node_encoders:
            self.node_encoders[node_type] = NodeEncoder(param_dim, child_dim, self.latent_dim)
            self.optimizers[node_type] = AdamW(self.node_encoders[node_type].parameters(), lr=self.lr)

        encoder = self.node_encoders[node_type]

        # Ensure child_latent is at least shape (1, child_dim)
        if child_latent.numel() == 0:
            child_latent = torch.zeros(1, 0)

        # Forward pass
        recon, z = encoder(params.unsqueeze(0), child_latent)
        return z.squeeze(0)

    def train(self, dataset: List[Shape3D], epochs: int = 50):
        criterion = nn.MSELoss()
        for epoch in range(epochs):
            total_loss = 0.0
            for node in dataset:
                # Encode recursively and compute reconstruction loss per node type
                def train_node(node):
                    child_latents = []
                    for child in getattr(node, "children", []):
                        child_latents.append(train_node(child))
                    child_latent = torch.cat(child_latents) if child_latents else torch.zeros(0)
                    params = self.get_params_tensor(node)
                    param_dim = params.numel()
                    child_dim = child_latent.numel()

                    node_type = type(node).__name__
                    encoder = self.node_encoders[node_type]
                    optimizer = self.optimizers[node_type]

                    optimizer.zero_grad()
                    recon, z = encoder(params.unsqueeze(0), child_latent.unsqueeze(0))
                    target = torch.cat([params, child_latent]).unsqueeze(0)
                    loss = criterion(recon, target)
                    loss.backward()
                    optimizer.step()
                    return z.squeeze(0)

                total_loss += train_node(node).sum().item()
            print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(dataset):.6f}")

    def encode_dataset(self, dataset: List[Shape3D]) -> List[torch.Tensor]:
        """Return latent vectors for all shapes in dataset."""
        latents = []
        for node in dataset:
            z = self.encode_node(node)
            latents.append(z)
        return latents

In [10]:
recursive_abstraction = RecursiveAbstraction(latent_dim=32)
recursive_abstraction.train(dataset, epochs=30)

# Encode entire dataset
shape_latents = recursive_abstraction.encode_dataset(dataset)
print(f"Encoded {len(shape_latents)} shapes into latent vectors of dim {shape_latents[0].shape[0]}")

KeyError: 'Rect3D'